# Quant Researcher — Practical Lab 3
### Options, Greeks & VaR — Part 5 (+ Part 10A Binomial) ka practical version


In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from scipy.stats import norm
np.random.seed(11)
plt.rcParams['figure.figsize']=(8,4)


## A. Black-Scholes — scratch se likho, finite-difference se check karo

Formula yaad rakhna kaafi nahi — interview mein "code it up" bol sakte hain. Delta ko do tarike se nikaalte hain: closed-form formula se, aur numerically (price ko S thoda hila ke) — dono match karne chahiye.


In [ ]:
def bs_price(S, K, T, r, sigma, kind='call'):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    if kind == 'call':
        return S*norm.cdf(d1) - K*np.exp(-r*T)*norm.cdf(d2)
    else:
        return K*np.exp(-r*T)*norm.cdf(-d2) - S*norm.cdf(-d1)

S, K, T, r, sigma = 100, 100, 0.5, 0.06, 0.25
call_price = bs_price(S, K, T, r, sigma, 'call')
put_price  = bs_price(S, K, T, r, sigma, 'put')
print(f'Call: {call_price:.4f}   Put: {put_price:.4f}')

# Put-Call Parity check (Part 1B-L)
lhs = call_price - put_price
rhs = S - K*np.exp(-r*T)
print(f'Put-Call parity check  (C - P) = {lhs:.4f}  vs  (S - K*e^-rT) = {rhs:.4f}')


## B. The Greeks — closed-form aur finite-difference, dono

Agar kabhi koi Greek formula bhool jao interview mein, finite-difference fallback hamesha kaam aata hai — bas price function ko thoda perturb karo.


In [ ]:
def bs_greeks(S, K, T, r, sigma, kind='call'):
    d1 = (np.log(S/K) + (r + 0.5*sigma**2)*T) / (sigma*np.sqrt(T))
    d2 = d1 - sigma*np.sqrt(T)
    delta = norm.cdf(d1) if kind=='call' else norm.cdf(d1)-1
    gamma = norm.pdf(d1) / (S*sigma*np.sqrt(T))                 # same for call & put
    vega  = S*norm.pdf(d1)*np.sqrt(T) / 100                      # per 1% vol move
    if kind == 'call':
        theta = (-S*norm.pdf(d1)*sigma/(2*np.sqrt(T)) - r*K*np.exp(-r*T)*norm.cdf(d2)) / 365
        rho   = K*T*np.exp(-r*T)*norm.cdf(d2) / 100
    else:
        theta = (-S*norm.pdf(d1)*sigma/(2*np.sqrt(T)) + r*K*np.exp(-r*T)*norm.cdf(-d2)) / 365
        rho   = -K*T*np.exp(-r*T)*norm.cdf(-d2) / 100
    return dict(delta=delta, gamma=gamma, vega=vega, theta=theta, rho=rho)

greeks = bs_greeks(S, K, T, r, sigma, 'call')
print({k: round(v,4) for k,v in greeks.items()})

# finite-difference check on delta and gamma
eps = 0.01
price_up = bs_price(S+eps, K, T, r, sigma, 'call')
price_dn = bs_price(S-eps, K, T, r, sigma, 'call')
delta_fd = (price_up - price_dn) / (2*eps)
gamma_fd = (price_up - 2*call_price + price_dn) / eps**2
print(f'delta: formula={greeks["delta"]:.5f}  finite-diff={delta_fd:.5f}')
print(f'gamma: formula={greeks["gamma"]:.5f}  finite-diff={gamma_fd:.5f}')


## Part 10A — Binomial Option Pricing (converges to Black-Scholes)

American options ke liye binomial zaroori hota hai (early exercise), European ke liye ye bas BS ka discrete-time version hai — jitna zyada steps utna close.


In [ ]:
def binomial_price(S, K, T, r, sigma, n=300, kind='call', american=False):
    dt = T/n
    u = np.exp(sigma*np.sqrt(dt)); d = 1/u
    p = (np.exp(r*dt) - d) / (u - d)
    prices = S * u**np.arange(n,-1,-1) * d**np.arange(0,n+1)
    values = np.maximum(prices-K,0) if kind=='call' else np.maximum(K-prices,0)
    for i in range(n-1,-1,-1):
        prices = prices[:i+1] / u   # step back one level (not strictly needed if only pricing European)
        continuation = np.exp(-r*dt)*(p*values[:-1] + (1-p)*values[1:])
        if american:
            exercise = np.maximum(prices-K,0) if kind=='call' else np.maximum(K-prices,0)
            values = np.maximum(continuation, exercise)
        else:
            values = continuation
    return values[0]

for n in [10, 50, 300]:
    print(f'n={n:4d} steps  ->  binomial call = {binomial_price(S,K,T,r,sigma,n):.4f}   (BS = {call_price:.4f})')

american_put = binomial_price(S, K, T, r, sigma, n=300, kind='put', american=True)
european_put = binomial_price(S, K, T, r, sigma, n=300, kind='put', american=False)
print(f'American put = {american_put:.4f}  vs  European put = {european_put:.4f}  (American >= European, early exercise value)')


## C. Four VaR Methods — Historical, Parametric, Monte Carlo, Cornish-Fisher

Ek hi portfolio pe chaaron method lagate hain taaki unka farak khud dikhe.


In [ ]:
np.random.seed(5)
# fat-tailed, slightly negatively skewed returns -- realistic for equity portfolios
daily_returns = pd.Series(np.random.standard_t(df=5, size=2000) * 0.012 - 0.0005)

def historical_var(returns, alpha=0.05):
    return -np.percentile(returns, alpha*100)

def parametric_var(returns, alpha=0.05):
    mu, sigma = returns.mean(), returns.std()
    return -(mu + norm.ppf(alpha)*sigma)

def monte_carlo_var(returns, alpha=0.05, n_sims=50000):
    mu, sigma = returns.mean(), returns.std()
    simulated = np.random.normal(mu, sigma, n_sims)
    return -np.percentile(simulated, alpha*100)

def cornish_fisher_var(returns, alpha=0.05):
    mu, sigma = returns.mean(), returns.std()
    skew = ((returns-mu)**3).mean() / sigma**3
    kurt = ((returns-mu)**4).mean() / sigma**4 - 3     # excess kurtosis
    z = norm.ppf(alpha)
    z_cf = z + (z**2-1)*skew/6 + (z**3-3*z)*kurt/24 - (2*z**3-5*z)*skew**2/36
    return -(mu + z_cf*sigma)

for name, fn in [('Historical', historical_var), ('Parametric', parametric_var),
                  ('Monte Carlo', monte_carlo_var), ('Cornish-Fisher', cornish_fisher_var)]:
    print(f'{name:15s} 1-day 95% VaR: {fn(daily_returns):.4f}')


**Interpretation:** Parametric VaR normal-distribution assume karta hai, isliye fat tails ko underestimate karta hai. Cornish-Fisher skew/kurtosis ke liye adjust karta hai, isliye Historical ke zyada close aata hai — yahi reason hai Cornish-Fisher practice mein zyada trusted hai jab returns normal nahi hote (jo almost hamesha hota hai).


## D. CVaR / Expected Shortfall

VaR sirf ek threshold batata hai ("95% confidence se itna loss"), CVaR batata hai "agar wo 5% tail event ho hi gaya, to average kitna loss hoga" — zyada informative, especially fat tails mein.


In [ ]:
def historical_cvar(returns, alpha=0.05):
    var_threshold = -historical_var(returns, alpha)
    tail_losses = returns[returns <= var_threshold]
    return -tail_losses.mean()

print(f'Historical VaR  : {historical_var(daily_returns):.4f}')
print(f'Historical CVaR : {historical_cvar(daily_returns):.4f}   (hamesha >= VaR)')


## E. Kupiec Backtesting — VaR model khud honest hai ya nahi

Agar 95% VaR sahi calibrated hai, to roz ke returns mein se sirf ~5% hi VaR threshold se zyada loss dikhane chahiye. Kupiec test ye formally check karta hai.


In [ ]:
from scipy.stats import chi2

def kupiec_test(returns, var_series, alpha=0.05):
    breaches = (returns < -var_series).sum()
    n = len(returns)
    pi_hat = breaches / n
    if pi_hat in (0,1):
        return breaches, n, np.nan
    lr = -2*np.log(((1-alpha)**(n-breaches) * alpha**breaches) /
                    ((1-pi_hat)**(n-breaches) * pi_hat**breaches))
    p_value = 1 - chi2.cdf(lr, df=1)
    return breaches, n, p_value

# static VaR applied over the whole sample (simplification for this exercise)
static_var = parametric_var(daily_returns)
var_series = pd.Series(static_var, index=daily_returns.index)
breaches, n, p_value = kupiec_test(daily_returns, var_series)

print(f'Breaches: {breaches} / {n}  (expected ~{0.05*n:.0f} at 95% VaR)')
print(f'Kupiec test p-value: {p_value:.4f}')
print('VaR model NOT rejected (looks well-calibrated)' if p_value > 0.05 else 'VaR model REJECTED — breach rate is significantly off from 5%')


## F. Stress Testing & Scenario Analysis

VaR "normal" market conditions assume karta hai. Stress test specific, extreme scenarios apply karta hai — jaise "2008-style -8% single day shock" — aur dekhta hai portfolio ka kya hota hai, bina kisi probability assumption ke.


In [ ]:
def stress_test_option_position(S, K, T, r, sigma, shock_pct, vol_shock=0.0, kind='call'):
    S_shocked = S*(1+shock_pct)
    sigma_shocked = sigma + vol_shock
    base = bs_price(S, K, T, r, sigma, kind)
    shocked = bs_price(S_shocked, K, T, r, sigma_shocked, kind)
    return shocked - base

scenarios = {
    '2008-style crash (-8%, vol +15pts)': (-0.08, 0.15),
    'Flash crash (-5%, vol +25pts)':      (-0.05, 0.25),
    'Melt-up (+6%, vol -5pts)':           (0.06, -0.05),
}

for name, (shock, vol_shock) in scenarios.items():
    pnl = stress_test_option_position(S, K, T, r, sigma, shock, vol_shock, 'call')
    print(f'{name:38s} -> option P&L: {pnl:+.4f}')


**Kya dekhna hai:** notice karo ki crash scenario mein call ka delta exposure loss deta hai, lekin vol spike (vega) kuch loss offset karta hai — yehi wo interaction hai jo sirf VaR se nahi dikhta, scenario analysis se dikhta hai.


## G/H — End-to-end summary

Agar live mein VaR/options pe pucha jaaye: price karo (BS ya binomial) → Greeks nikaalo (formula ya finite-diff) → multiple VaR methods compare karo → Kupiec se apna model check karo → stress scenarios se tail risk dekho. Yahi 5-step flow Part 5-G ke "VaR/Stress-Testing Project" ka practical skeleton hai.
